In [1]:
from vllm import LLM, SamplingParams
from reasoning.tools.utils import load_model_with_vllm
LM = load_model_with_vllm('Qwen/QwQ-32B', task = 'auto', tensor_parallel_size = 4, gpu_memory_utilization = 0.9)

INFO 03-20 15:43:00 __init__.py:183] Automatically detected platform cuda.
INFO 03-20 15:43:06 config.py:526] This model supports multiple tasks: {'reward', 'generate', 'score', 'classify', 'embed'}. Defaulting to 'generate'.
INFO 03-20 15:43:07 config.py:1383] Defaulting to use mp for distributed inference
WARNING 03-20 15:43:07 arg_utils.py:1119] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 03-20 15:43:07 config.py:1538] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 03-20 15:43:07 llm_engine.py:232] Initializing a V0 LLM engine (v0.7.1) with config: model='Qwen/QwQ-32B', speculative_config=None, tokenizer='Qwen/QwQ-32B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=Fals

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]


(VllmWorkerProcess pid=2420657) INFO 03-20 15:43:11 weight_utils.py:251] Using model weights format ['*.safetensors']
(VllmWorkerProcess pid=2420648) INFO 03-20 15:43:32 model_runner.py:1116] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=2420652) INFO 03-20 15:43:38 model_runner.py:1116] Loading model weights took 15.3937 GB
INFO 03-20 15:43:40 model_runner.py:1116] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=2420657) INFO 03-20 15:43:42 model_runner.py:1116] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=2420657) INFO 03-20 15:43:45 worker.py:266] Memory profiling takes 3.08 seconds
(VllmWorkerProcess pid=2420657) INFO 03-20 15:43:45 worker.py:266] the current vLLM instance can use total_gpu_memory (47.43GiB) x gpu_memory_utilization (0.90) = 42.69GiB
(VllmWorkerProcess pid=2420652) (VllmWorkerProcess pid=2420657) INFO 03-20 15:43:45 worker.py:266] Memory profiling takes 3.08 seconds
INFO 03-20 15:43:45 worker.py:266] model weights take 15

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

(VllmWorkerProcess pid=2420652) (VllmWorkerProcess pid=2420648) INFO 03-20 15:43:49 model_runner.py:1435] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 03-20 15:43:49 model_runner.py:1435] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.


Capturing CUDA graph shapes:  97%|█████████▋| 34/35 [00:18<00:00,  2.09it/s]

(VllmWorkerProcess pid=2420648) INFO 03-20 15:44:09 model_runner.py:1563] Graph capturing finished in 19 secs, took 0.76 GiB


Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:19<00:00,  1.80it/s]

INFO 03-20 15:44:09 model_runner.py:1563] Graph capturing finished in 19 secs, took 0.76 GiB
(VllmWorkerProcess pid=2420657) INFO 03-20 15:44:09 model_runner.py:1563] Graph capturing finished in 19 secs, took 0.76 GiB
(VllmWorkerProcess pid=2420652) INFO 03-20 15:44:09 model_runner.py:1563] Graph capturing finished in 19 secs, took 0.76 GiB
INFO 03-20 15:44:09 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 26.67 seconds


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 加载模型和分词器
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # 使用半精度以减少内存使用
    device_map="cuda:0"  # 使用第一张GPU
)

# 准备输入提示
prompt = "用中文介绍一下人工智能的应用"

# 对输入进行编码
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 使用generate()方法生成文本
outputs = model.generate(
    **inputs,
    max_new_tokens=200,  # 生成的最大新token数
    do_sample=True,      # 使用采样而不是贪婪解码
    temperature=0.7,     # 温度参数，控制随机性
    top_p=0.9,           # nucleus sampling参数
    repetition_penalty=1.1  # 重复惩罚
)

# 解码输出
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 打印结果
print(f"输入: {prompt}")
print(f"输出: {generated_text}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


输入: 用中文介绍一下人工智能的应用
输出: 用中文介绍一下人工智能的应用场景
人工智能（Artificial Intelligence，简称AI）是一种利用计算机系统来模拟人类思维、学习和决策能力的技术。人工智能的应用场景广泛，包括但不限于以下几个方面：
1.  **自然语言理解**：人工智能可以帮助我们更好地理解和分析自然语言中的意图、情感和语气，例如在 chatbot 和 virtual assistant 中使用。
2.  ** computer vision**：人工智能可以帮助我们识别和分析图像中的对象、形状和模式，从而实现自动驾驶、面部识别等应用。
3.  ** 语音识别**：人工智能可以帮助我们识别和分析语音中的内容，例如在 speech-to-text 和 voice assistants 中使用。
4.  **推荐系统**：人工智能可以帮助我们为用户提供个性化的推荐，例如


In [4]:
# load the libraies dynamcally so that no need to restart

from reasoning.models.model import Model
from reasoning.VMCTS.task import VMCTS_Task
from reasoning.tools.utils import seed_everything
seed_everything(110)
from reasoning.models.model import ValueModel_shepherd, ValueModel_qwen
from reasoning.evaluator.math_grader import math_equal, extract_answer


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [5]:
# load modesl from huggingface
policy_model = Model('meta-llama/Llama-3.2-3B-Instruct',device='cuda:0')
# reward_model = ValueModel_qwen('cuda:1',low = 0)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
policy_model.model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm

In [9]:
import torch
def generate_text(model, tokenizer, prompt, max_length=1024):
    # 准备输入
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # 生成文本
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_return_sequences=1,
            temperature=0.7,
            top_p=0.9,
        )
    
    # 解码输出
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

In [ ]:
prompt = "How many positive whole-number divisors does 196 have?"
generated_text = generate_text(policy_model.model, policy_model.tokenizer, prompt)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [11]:
generated_text

'How many positive whole-number divisors does 196 have??\n## Step 1: Factorize the number 196\nTo find the number of positive whole-number divisors of 196, we first need to factorize 196 into its prime factors.\n\n## Step 2: Determine the prime factorization of 196\nThe prime factorization of 196 is 2^2 * 7^2, since 196 can be divided evenly by 2 twice and by 7 twice.\n\n## Step 3: Apply the formula to find the number of divisors\nThe formula to find the number of divisors for a number given its prime factorization is (a+1)(b+1), where a and b are the powers of the prime factors. In this case, a=2 and b=2.\n\n## Step 4: Calculate the number of divisors\nUsing the formula, the number of divisors of 196 is (2+1)(2+1) = 3*3 = 9.\n\nThe final answer is: $\\boxed{9}$'

In [10]:
prompt = "How many positive whole-number divisors does 196 have?"
# policy_model.model = torch.compile(policy_model.model)
import time
start_time = time.time()
response = generate_text(policy_model.model, policy_model.tokenizer, prompt)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(response)



AttributeError: 'function' object has no attribute 'generate'

In [ ]:
from reasoning.models.model_vllm import LlamaPolicy, QwenReward

policy_model = LlamaPolicy(model_name = 'meta-llama/Llama-3.2-3B-Instruct', gpu_memory_utilization = 0.4)






INFO 03-06 14:38:57 __init__.py:183] Automatically detected platform cuda.
INFO 03-06 14:39:04 config.py:526] This model supports multiple tasks: {'score', 'classify', 'reward', 'generate', 'embed'}. Defaulting to 'generate'.
INFO 03-06 14:39:04 config.py:1383] Defaulting to use mp for distributed inference
WARNING 03-06 14:39:04 arg_utils.py:1119] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 03-06 14:39:04 config.py:1538] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 03-06 14:39:04 llm_engine.py:232] Initializing a V0 LLM engine (v0.7.1) with config: model='meta-llama/Llama-3.2-3B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokeniz

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(VllmWorkerProcess pid=3180470) INFO 03-06 14:39:10 model_runner.py:1116] Loading model weights took 1.5346 GB
(VllmWorkerProcess pid=3180465) INFO 03-06 14:39:10 model_runner.py:1116] Loading model weights took 1.5346 GB
(VllmWorkerProcess pid=3180462) INFO 03-06 14:39:10 model_runner.py:1116] Loading model weights took 1.5346 GB
INFO 03-06 14:39:10 model_runner.py:1116] Loading model weights took 1.5346 GB
(VllmWorkerProcess pid=3180465) INFO 03-06 14:39:13 worker.py:266] Memory profiling takes 2.52 seconds
(VllmWorkerProcess pid=3180462) (VllmWorkerProcess pid=3180465) INFO 03-06 14:39:13 worker.py:266] Memory profiling takes 2.52 seconds
INFO 03-06 14:39:13 worker.py:266] the current vLLM instance can use total_gpu_memory (47.43GiB) x gpu_memory_utilization (0.40) = 18.97GiB
(VllmWorkerProcess pid=3180470) (VllmWorkerProcess pid=3180462) (VllmWorkerProcess pid=3180465) INFO 03-06 14:39:13 worker.py:266] Memory profiling takes 2.53 seconds
INFO 03-06 14:39:13 worker.py:266] the curr

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 03-06 14:39:20 model_runner.py:1435] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.


Capturing CUDA graph shapes:  97%|█████████▋| 34/35 [00:14<00:00,  2.43it/s]

(VllmWorkerProcess pid=3180465) INFO 03-06 14:39:34 model_runner.py:1563] Graph capturing finished in 15 secs, took 0.40 GiB
(VllmWorkerProcess pid=3180462) INFO 03-06 14:39:34 model_runner.py:1563] Graph capturing finished in 15 secs, took 0.40 GiB
(VllmWorkerProcess pid=3180470) INFO 03-06 14:39:34 model_runner.py:1563] Graph capturing finished in 15 secs, took 0.40 GiB


Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:14<00:00,  2.37it/s]

INFO 03-06 14:39:34 model_runner.py:1563] Graph capturing finished in 15 secs, took 0.40 GiB
INFO 03-06 14:39:34 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 24.40 seconds


In [9]:
message = "How many positive whole-number divisors does 196 have?"
import time
start_time = time.time()    
text = policy_model.model.generate(message, sampling_params=policy_model.sampling_params)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
answer = text[0].outputs[0].text.strip().split('\n\n')
print(answer)
print(sum(len(part) for part in answer))




Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s, est. speed input: 15.74 toks/s, output: 154.97 toks/s]

Time taken: 0.830190896987915 seconds
['?\n## Step 1: Factorize the number 196\nFirst, we need to factorize 196 into its prime factors to find its divisors.', '## Step 2: Find the prime factorization of 196\nThe prime factorization of 196 is 2^2 * 7^2.', '## Step 3: Apply the formula for the number of divisors\nThe formula for the number of divisors of a number is to add 1 to each exponent in the prime factorization and then multiply these results together. For 196 = 2^2 * 7^2, the number of div']
448


[E306 14:48:56.936646017 socket.cpp:1011] [c10d] The client socket has timed out after 600000ms while trying to connect to (10.138.119.8, 33257).
[W306 14:48:56.939857980 TCPStore.cpp:358] [c10d] TCP client failed to connect/validate to host 10.138.119.8:33257 - retrying (try=0, timeout=600000ms, delay=33750ms): The client socket has timed out after 600000ms while trying to connect to (10.138.119.8, 33257).
Exception raised from throwTimeoutError at ../torch/csrc/distributed/c10d/socket.cpp:1013 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x96 (0x7f0618346446 in /ssdscratch/byuan48/software/anaconda3/envs/mcts/lib/python3.12/site-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x15e04c6 (0x7f064e97c4c6 in /ssdscratch/byuan48/software/anaconda3/envs/mcts/lib/python3.12/site-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x6029d95 (0x7f06533c5d95 in /ssdscratch/byuan48/software/anaconda3/envs/mcts/lib/python3.

In [10]:
# import model from hugginface and run inference on it
from reasoning.models.model import Model

model = Model('meta-llama/Llama-3.2-3B-Instruct',device='cuda:0')

ImportError: Using `low_cpu_mem_usage=True` or a `device_map` requires Accelerate: `pip install 'accelerate>=0.26.0'`

In [ ]:
# run inference on the model and also check how fast it is
question = "How many positive whole-number divisors does 196 have?"
answer = '$\\boxed{9}$'


In [ ]:
question = "How many positive whole-number divisors does 196 have?"
answer = '$\\boxed{9}$'
task = VMCTS_Task(question, answer = answer, propose_method=policy_model, value_method=reward_model, iteration_limit=50, end_gate=0.95, branch=2)
output = task.run()

<beging new round, current round:0>

----------------------------------------
selection phase

selected node: 
----------------------------------------
expansion phase


 ============================== proposal ============================== 
step:  1
standardized next step: Step 1: Understand the problem To find the number of positive whole-number divisors of 196, we need to factorize 196 into its prime factors.  ##


 ============================== proposal ============================== 
step:  1
standardized next step: Step 1: The first step in solving this problem is to find the prime factorization of 196, which is the expression of 196 as a product of its prime factors.


 ============================== proposal ============================== 
step:  1
standardized next step: Step 1: To find the number of positive whole-number divisors of 196, we need to factorize 196 into its prime factors.

获得评分:0.99609375

获得评分:0.99609375

获得评分:1.0

----------------------------------------
sim

In [9]:
from reasoning.MCTS.task import MCTS_Task
question = "How many positive whole-number divisors does 196 have?"
answer = '$\\boxed{9}$'
task = MCTS_Task(question, answer = answer, propose_method=policy_model, value_method=reward_model, iteration_limit=50, end_gate=0.95, branch=2)
output = task.run()

<beging new round, current round:0>

----------------------------------------
selection phase

selected node: 
----------------------------------------
expansion phase


 ============================== proposal ============================== 
step:  1
standardized next step: Step 1: Understand the problem The problem asks us to find the number of positive whole-number divisors of the number 196.  ##


 ============================== proposal ============================== 
step:  1
standardized next step: Step 1: Understand the problem We need to find the number of positive whole-number divisors of 196.  ##

获得评分:1.0

获得评分:1.0

----------------------------------------
simulation phase


 ============================== proposal ============================== 
step:  2
next step is repeated！


 ============================== proposal ============================== 
step:  2
standardized next step: Step 2: First, let's find the prime factorization of 196. We can start by dividing 196 by t